# Compare two TPC gain maps

If the two maps are inverse corrections, their **bin-by-bin product should be close to 1**.
This notebook draws the two maps, their product, `product - 1`, and averaged X/Y projections.


In [ ]:
import ROOT
ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetPalette(ROOT.kViridis)
%jsroot on
# ============================================================
# EDIT ONLY THIS BLOCK
# ============================================================
file_fullmap = "input/dedx_full_maps_79513_SA.root"
file_gainmap = "../distort/input/layer_gain_79513_Mariia_side01.root"

hist_fullmap = "Full_map_s0"
hist_gainmap = "hGainMap_side0_South"   # e.g. hGainMap_side0_South
# ============================================================

f1 = ROOT.TFile.Open(file_fullmap)
f2 = ROOT.TFile.Open(file_gainmap)

if not f1 or f1.IsZombie():
    raise RuntimeError(f"Cannot open {file_fullmap}")
if not f2 or f2.IsZombie():
    raise RuntimeError(f"Cannot open {file_gainmap}")

h1 = f1.Get(hist_fullmap)
h2 = f2.Get(hist_gainmap)

if not h1:
    raise RuntimeError(f"Cannot find {hist_fullmap} in {file_fullmap}")
if not h2:
    raise RuntimeError(f"Cannot find {hist_gainmap} in {file_gainmap}")

# Detach copies from the ROOT files.
A = h1.Clone("A_full_map")
B = h2.Clone("B_gain_map")
A.SetDirectory(0)
B.SetDirectory(0)

print("A:", A.GetName(), A.GetNbinsX(), "x", A.GetNbinsY())
print("B:", B.GetName(), B.GetNbinsX(), "x", B.GetNbinsY())

if A.GetNbinsX() != B.GetNbinsX() or A.GetNbinsY() != B.GetNbinsY():
    raise RuntimeError("The maps do not have the same number of X/Y bins.")

Welcome to JupyROOT 6.30/06
A: A_full_map 12 x 48
B: B_gain_map 12 x 48


In [ ]:
6     const std::string& garfield_output = "output/simple_2d_rossegger.root",
 17     const bool use_real_tpc_source_geometry = true,
 18     const bool use_frame_charge_model = false,
 19     const double frame_reference_phi = 0.0,
 20     const std::string& pad_placement_file = ""include/TPC_pad_placement.txt"",
 21     const std::string& gain_map_file = "include/layer_gain_79513_Mariia_side01.root"

In [2]:
# Bin-by-bin product.  If A and B are inverse maps, this should be ~1.
A.Scale(1.0 / A.Integral()*12*48)
B.Scale(1.0 / B.Integral()*12*48)
P = A.Clone("product")
P.SetDirectory(0)
P.Multiply(B)
P.SetTitle("A #times B;X;Y")

# Product minus unity makes small mismatches much easier to see.
D = P.Clone("product_minus_one")
D.SetDirectory(0)
D.SetTitle("A #times B - 1;X;Y")

for ix in range(1, D.GetNbinsX() + 1):
    for iy in range(1, D.GetNbinsY() + 1):
        D.SetBinContent(ix, iy, D.GetBinContent(ix, iy) - 1.0)

# Summary, excluding under/overflow bins.
vals = []
for ix in range(1, P.GetNbinsX() + 1):
    for iy in range(1, P.GetNbinsY() + 1):
        vals.append(P.GetBinContent(ix, iy))

import numpy as np
vals = np.asarray(vals)
print(f"Product mean = {vals.mean():.6g}")
print(f"Product RMS  = {vals.std():.6g}")
print(f"Product min  = {vals.min():.6g}")
print(f"Product max  = {vals.max():.6g}")
print(f"Mean |P-1|  = {np.mean(np.abs(vals-1.0)):.6g}")

Product mean = 0.978462
Product RMS  = 0.100985
Product min  = 0.315643
Product max  = 1.167
Mean |P-1|  = 0.0758284


In [3]:
# ------------------------------------------------------------
# 2D inspection
# ------------------------------------------------------------
c2d = ROOT.TCanvas("c2d", "2D gain-map comparison", 1500, 1100)
c2d.Divide(2, 2)

c2d.cd(1)
ROOT.gPad.SetRightMargin(0.15)
A.SetTitle("Map A: full_map_s;X;Y")
A.Draw("COLZ")

c2d.cd(2)
ROOT.gPad.SetRightMargin(0.15)
B.SetTitle("Map B: hGainMap;X;Y")
B.Draw("COLZ")

c2d.cd(3)
ROOT.gPad.SetRightMargin(0.15)
P.SetTitle("Bin-by-bin product A #times B;X;Y")
P.Draw("COLZ")

c2d.cd(4)
ROOT.gPad.SetRightMargin(0.15)
D.SetTitle("Residual product A #times B - 1;X;Y")
D.Draw("COLZ")

c2d.Draw()

In [4]:
# Optional numerical values written on top of the product map.
# This is useful only if the map does not have too many bins.
ctext = ROOT.TCanvas("ctext", "Product with bin values", 1100, 850)
ctext.SetRightMargin(0.15)

ROOT.gStyle.SetPaintTextFormat(".3f")
P.SetMarkerSize(0.7)
P.SetTitle("A #times B (numbers are bin contents);X;Y")
P.Draw("COLZ TEXT")

ctext.Draw()

In [5]:
# ------------------------------------------------------------
# Average projections rather than ROOT's default SUM projection.
# This keeps the gain scale meaningful.
# ------------------------------------------------------------
Ax = A.ProjectionX("Ax")
Bx = B.ProjectionX("Bx")
Px = P.ProjectionX("Px")

Ay = A.ProjectionY("Ay")
By = B.ProjectionY("By")
Py = P.ProjectionY("Py")

for h in (Ax, Bx, Px):
    h.SetDirectory(0)
    h.Scale(1.0 / A.GetNbinsY())

for h in (Ay, By, Py):
    h.SetDirectory(0)
    h.Scale(1.0 / A.GetNbinsX())

# Styling
Ax.SetLineColor(ROOT.kBlue + 1)
Ax.SetLineWidth(2)
Bx.SetLineColor(ROOT.kRed + 1)
Bx.SetLineWidth(2)

Ay.SetLineColor(ROOT.kBlue + 1)
Ay.SetLineWidth(2)
By.SetLineColor(ROOT.kRed + 1)
By.SetLineWidth(2)

Px.SetLineColor(ROOT.kBlack)
Px.SetLineWidth(2)
Py.SetLineColor(ROOT.kBlack)
Py.SetLineWidth(2)

cproj = ROOT.TCanvas("cproj", "Gain-map projections", 1500, 1000)
cproj.Divide(2, 2)

cproj.cd(1)
Ax.SetTitle("Average projection on X;X;average bin content")
Ax.Draw("HIST")
Bx.Draw("HIST SAME")
legx = ROOT.TLegend(0.68, 0.75, 0.88, 0.88)
legx.AddEntry(Ax, "full_map_s", "l")
legx.AddEntry(Bx, "hGainMap", "l")
legx.Draw()

cproj.cd(2)
Ay.SetTitle("Average projection on Y;Y;average bin content")
Ay.Draw("HIST")
By.Draw("HIST SAME")
legy = ROOT.TLegend(0.68, 0.75, 0.88, 0.88)
legy.AddEntry(Ay, "full_map_s", "l")
legy.AddEntry(By, "hGainMap", "l")
legy.Draw()

cproj.cd(3)
Px.SetTitle("Average product vs X;X;<A #times B>")
Px.Draw("HIST")
one_x = ROOT.TLine(Px.GetXaxis().GetXmin(), 1.0,
                   Px.GetXaxis().GetXmax(), 1.0)
one_x.SetLineStyle(2)
one_x.Draw()

cproj.cd(4)
Py.SetTitle("Average product vs Y;Y;<A #times B>")
Py.Draw("HIST")
one_y = ROOT.TLine(Py.GetXaxis().GetXmin(), 1.0,
                   Py.GetXaxis().GetXmax(), 1.0)
one_y.SetLineStyle(2)
one_y.Draw()

cproj.Draw()

In [6]:
# Optional: distribution of the product itself.
# A narrow peak at 1 means the two maps are very nearly exact inverses.
hdist = ROOT.TH1D("hdist", "Distribution of A #times B;A #times B;bins",
                  200, 0.8, 1.2)

for ix in range(1, P.GetNbinsX() + 1):
    for iy in range(1, P.GetNbinsY() + 1):
        hdist.Fill(P.GetBinContent(ix, iy))

cdist = ROOT.TCanvas("cdist", "Product distribution", 900, 700)
hdist.SetLineWidth(2)
hdist.Draw("HIST")

line1 = ROOT.TLine(1.0, 0.0, 1.0, 1.05 * hdist.GetMaximum())
line1.SetLineStyle(2)
line1.Draw()

cdist.Draw()